In [1]:
#رح نجيب داتا للمشروع من موقع ال The Hacker News


In [2]:
pip install requests beautifulsoup4

In [3]:
import requests
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

article_links = set()
target_articles_count = 60
url = "https://thehackernews.com"

print("Tracking Now ")

while len(article_links) < target_articles_count and url:
    print(f"Checking Deal Now  {url[:100]}...")
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print(f"Error In the connection  {res.status_code}")
        break

    soup = BeautifulSoup(res.text, "html.parser")

    # 1. جمع روابط المقالات (تحمل دائماً كلاس story-link)
    posts = soup.find_all("a", class_="story-link")
    for post in posts:
        link = post.get("href")
        if link and "thehackernews.com/20" in link:
            article_links.add(link)
            if len(article_links) >= target_articles_count:
                break

    # 2. استخراج رابط الصفحة التالية بدقة
    # الموقع يضع رابط الصفحة التالية داخل div يحمل id="blog-pager"
    blog_pager = soup.find("div", id="blog-pager")
    next_link = None
    if blog_pager:
        # البحث عن أي رابط داخل الـ pager يشير إلى صفحة أقدم
        older_tag = blog_pager.find("a", class_=lambda c: c and "older" in c.lower())
        if not older_tag:
            older_tag = blog_pager.find("a", href=lambda h: h and "updated-max" in h)
        if older_tag:
            next_link = older_tag.get("href")

    if next_link and next_link != url:
        url = next_link
        time.sleep(1)
    else:
        break

article_links = list(article_links)
print("-" * 50)
print(f"The Num Of Links  Is: {len(article_links)}")

Tracking Now 
Checking Deal Now  https://thehackernews.com...
Checking Deal Now  https://thehackernews.com/search?updated-max=2026-10-16T01:54:00%2B05:30&max-results=16...
Checking Deal Now  https://thehackernews.com/search?updated-max=2026-10-14T23:08:00%2B05:30&max-results=16&start=16&by-...
Checking Deal Now  https://thehackernews.com/search?updated-max=2026-10-13T22:58:00%2B05:30&max-results=16&start=32&by-...
--------------------------------------------------
The Num Of Links  Is: 60


In [4]:
all_articles_text = []

print(f"بدء سحب محتوى {len(article_links)} مقالاً")

for i, link in enumerate(article_links, 1):
    print(f"[{i}/{len(article_links)}] سحب: {link[:60]}...")
    try:
        res = requests.get(link, headers=headers, timeout=10)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, "html.parser")
            article_body = soup.find("div", class_="articlebody")

            if article_body:
                paragraphs = article_body.find_all("p")
                text = " ".join([p.get_text() for p in paragraphs if p.get_text()])
                if len(text.split()) > 50:
                    all_articles_text.append(text)
    except Exception as e:
        print(f"خطأ أثناء سحب المقال: {e}")

    time.sleep(1)

print("-" * 50)
print(f"{len(all_articles_text)} ")

بدء سحب محتوى 60 مقالاً
[1/60] سحب: https://thehackernews.com/2026/10/why-ai-confidence-is-not-v...
[2/60] سحب: https://thehackernews.com/2026/10/drupal-patches-critical-co...
[3/60] سحب: https://thehackernews.com/2026/10/nginx-fixes-5-flaws-includ...
[4/60] سحب: https://thehackernews.com/2026/10/lg-smart-tvs-vulnerable-to...
[5/60] سحب: https://thehackernews.com/2026/10/attackers-exploit-fortios-...
[6/60] سحب: https://thehackernews.com/2026/10/webinar-how-to-stop-ai-dri...
[7/60] سحب: https://thehackernews.com/2026/10/two-critical-marklogic-fla...
[8/60] سحب: https://thehackernews.com/2026/10/how-to-run-deepfake-readin...
[9/60] سحب: https://thehackernews.com/2026/10/cert-in-warns-ai-agents-ac...
[10/60] سحب: https://thehackernews.com/2026/10/researchers-used-public-op...
[11/60] سحب: https://thehackernews.com/2026/10/critical-pwnescape-linux-k...
[12/60] سحب: https://thehackernews.com/2026/10/uat-11985-uses-fake-event-...
[13/60] سحب: https://thehackernews.com/2026/10/palo-alto-netw

In [5]:
#هسا بدي اسحب النصوص من داخل الروابط


In [8]:
import time
import requests
from bs4 import BeautifulSoup

all_articles_text = []
total_links = len(article_links)

# نمر على كل الروابط الـ 60 كاملة
for i, link in enumerate(article_links, 1):
    print(f"[{i:02d}/{total_links}] جاري سحب المقال من: {link[:60]}...")

    try:
        res = requests.get(link, headers=headers, timeout=10)

        if res.status_code == 200:
            article_soup = BeautifulSoup(res.text, "html.parser")
            article_body = article_soup.find("div", class_="articlebody")

            if article_body:
                paragraphs = article_body.find_all("p")
                text = " ".join([p.get_text() for p in paragraphs if p.get_text()])

                if len(text.split()) > 40:
                    all_articles_text.append(text)
            else:
                print("لم يتم العثور على متن المقال")
    except Exception as e:
        print(f"حدث خطأ أثناء السحب: {e}")

    time.sleep(1)

print("-" * 50)
print(f"تم سحب {len(all_articles_text)} مقالاً بنجاح من أصل {total_links}!")

if all_articles_text:
    print("عينة من أول 300 حرف من أول مقال:")
    print(all_articles_text[0][:300])

[01/60] جاري سحب المقال من: https://thehackernews.com/2026/10/why-ai-confidence-is-not-v...
[02/60] جاري سحب المقال من: https://thehackernews.com/2026/10/drupal-patches-critical-co...
[03/60] جاري سحب المقال من: https://thehackernews.com/2026/10/nginx-fixes-5-flaws-includ...
[04/60] جاري سحب المقال من: https://thehackernews.com/2026/10/lg-smart-tvs-vulnerable-to...
[05/60] جاري سحب المقال من: https://thehackernews.com/2026/10/attackers-exploit-fortios-...
[06/60] جاري سحب المقال من: https://thehackernews.com/2026/10/webinar-how-to-stop-ai-dri...
[07/60] جاري سحب المقال من: https://thehackernews.com/2026/10/two-critical-marklogic-fla...
[08/60] جاري سحب المقال من: https://thehackernews.com/2026/10/how-to-run-deepfake-readin...
[09/60] جاري سحب المقال من: https://thehackernews.com/2026/10/cert-in-warns-ai-agents-ac...
[10/60] جاري سحب المقال من: https://thehackernews.com/2026/10/researchers-used-public-op...
[11/60] جاري سحب المقال من: https://thehackernews.com/2026/10/critical-pwnescape

In [9]:
all_articles_text[:1]

["AI is already part of digital forensics and incident response work, with or without a policy in place to govern it. The issue is that these tools are growing more confident in their own output faster than practitioners are developing reliable ways to verify it. Without governance models and validation standards in place, that gap compounds and rarely stays contained to a workstation: an unverified finding can end up in a forensic report, a deposition, or in court.\xa0 At SANS, we created two new AI frameworks this year to help address this challenge: the Digital Forensics & AI Investigations Framework, which we mapped to the trusted standards established by the Scientific Working Group on Digital Evidence (SWGDE), and the Incident Response & AI Investigations Framework, which we mapped to the NIST Cybersecurity Framework 2.0. Both frameworks rest on the same principle: AI can accelerate investigative work, but the human examiner remains the final authority at every phase.\xa0 In digi

Preprocessing


In [10]:
#هسا بدي اجمع المقالات ب مكان واحد

In [11]:
import re
from nltk.tokenize import word_tokenize


In [12]:

#بدي اجمع النص كامل
full_text =" ".join(all_articles_text)
len(full_text)
full_text_lower =full_text.lower()

cleaned_text =re.sub(r'[^a-z0-9\s]+',' ',full_text_lower)
cleaned_text


'ai is already part of digital forensics and incident response work  with or without a policy in place to govern it  the issue is that these tools are growing more confident in their own output faster than practitioners are developing reliable ways to verify it  without governance models and validation standards in place  that gap compounds and rarely stays contained to a workstation  an unverified finding can end up in a forensic report  a deposition  or in court \xa0 at sans  we created two new ai frameworks this year to help address this challenge  the digital forensics   ai investigations framework  which we mapped to the trusted standards established by the scientific working group on digital evidence  swgde  and the incident response   ai investigations framework  which we mapped to the nist cybersecurity framework 2 0  both frameworks rest on the same principle  ai can accelerate investigative work  but the human examiner remains the final authority at every phase \xa0 in digita

In [13]:
#Tokenization
from nltk.tokenize import word_tokenize




In [14]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [15]:
raw_tokens = word_tokenize(cleaned_text)
raw_tokens

['ai',
 'is',
 'already',
 'part',
 'of',
 'digital',
 'forensics',
 'and',
 'incident',
 'response',
 'work',
 'with',
 'or',
 'without',
 'a',
 'policy',
 'in',
 'place',
 'to',
 'govern',
 'it',
 'the',
 'issue',
 'is',
 'that',
 'these',
 'tools',
 'are',
 'growing',
 'more',
 'confident',
 'in',
 'their',
 'own',
 'output',
 'faster',
 'than',
 'practitioners',
 'are',
 'developing',
 'reliable',
 'ways',
 'to',
 'verify',
 'it',
 'without',
 'governance',
 'models',
 'and',
 'validation',
 'standards',
 'in',
 'place',
 'that',
 'gap',
 'compounds',
 'and',
 'rarely',
 'stays',
 'contained',
 'to',
 'a',
 'workstation',
 'an',
 'unverified',
 'finding',
 'can',
 'end',
 'up',
 'in',
 'a',
 'forensic',
 'report',
 'a',
 'deposition',
 'or',
 'in',
 'court',
 'at',
 'sans',
 'we',
 'created',
 'two',
 'new',
 'ai',
 'frameworks',
 'this',
 'year',
 'to',
 'help',
 'address',
 'this',
 'challenge',
 'the',
 'digital',
 'forensics',
 'ai',
 'investigations',
 'framework',
 'which',
 

In [16]:
tokens = [word for word in raw_tokens if len(word) > 1]
tokens[:20]

['ai',
 'is',
 'already',
 'part',
 'of',
 'digital',
 'forensics',
 'and',
 'incident',
 'response',
 'work',
 'with',
 'or',
 'without',
 'policy',
 'in',
 'place',
 'to',
 'govern',
 'it']

In [17]:
from collections import Counter
word_counts = Counter(tokens)
word_counts.most_common(20)

[('the', 3127),
 ('and', 1445),
 ('to', 1276),
 ('in', 807),
 ('of', 781),
 ('that', 655),
 ('it', 654),
 ('is', 542),
 ('on', 476),
 ('an', 377),
 ('for', 359),
 ('as', 316),
 ('with', 294),
 ('not', 256),
 ('or', 250),
 ('from', 237),
 ('its', 236),
 ('said', 235),
 ('has', 223),
 ('2026', 213)]

In [18]:
cleaned_tokens = [
    word for word in tokens
    #هل الكلمه كلها ارقام
    if not word.isdigit()
    #هل داخل هاي الكلمه في ارقام
    and not re.search(r'\d', word)
    #هل الكلمه متكرره اكثر من مرتين او لا
    and word_counts[word] >= 2

]
cleaned_tokens[:]

['ai',
 'is',
 'already',
 'part',
 'of',
 'digital',
 'forensics',
 'and',
 'incident',
 'response',
 'work',
 'with',
 'or',
 'without',
 'policy',
 'in',
 'place',
 'to',
 'it',
 'the',
 'issue',
 'is',
 'that',
 'these',
 'tools',
 'are',
 'growing',
 'more',
 'in',
 'their',
 'own',
 'output',
 'faster',
 'than',
 'practitioners',
 'are',
 'developing',
 'reliable',
 'ways',
 'to',
 'verify',
 'it',
 'without',
 'governance',
 'models',
 'and',
 'validation',
 'standards',
 'in',
 'place',
 'that',
 'gap',
 'and',
 'rarely',
 'stays',
 'contained',
 'to',
 'an',
 'finding',
 'can',
 'end',
 'up',
 'in',
 'forensic',
 'report',
 'or',
 'in',
 'court',
 'at',
 'sans',
 'we',
 'created',
 'two',
 'new',
 'ai',
 'frameworks',
 'this',
 'year',
 'to',
 'help',
 'address',
 'this',
 'challenge',
 'the',
 'digital',
 'forensics',
 'ai',
 'investigations',
 'framework',
 'which',
 'we',
 'mapped',
 'to',
 'the',
 'trusted',
 'standards',
 'established',
 'by',
 'the',
 'working',
 'group'

In [19]:
vocab =sorted(list(set(cleaned_tokens)))

vocab_size=len(vocab)
vocab_size

3441

In [20]:
word_to_ix ={word:i for i ,word in enumerate(vocab)}
ix_to_word ={i:word for i, word in enumerate(vocab)}
print(f"عدد الكلمات الكلي بعد التنظيف: {len(cleaned_tokens):,}")
print(f"حجم القاموس النظيف (vocab_size): {vocab_size} كلمة")
print("-" * 40)
print("عينة من أول 20 كلمة في القاموس الجديد:")
print(vocab[:20])

عدد الكلمات الكلي بعد التنظيف: 45,368
حجم القاموس النظيف (vocab_size): 3441 كلمة
----------------------------------------
عينة من أول 20 كلمة في القاموس الجديد:
['abandoned', 'abdulrahman', 'ability', 'able', 'about', 'above', 'abundance', 'abuse', 'abused', 'abusing', 'accelerate', 'accelerating', 'accelerator', 'accept', 'acceptable', 'accepted', 'accepts', 'access', 'accessed', 'accessibility']


In [21]:
import nltk
from nltk.corpus import stopwords
from collections import Counter
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

filtered_tokens = [
    w for w in tokens
    if w not in stop_words
    and not w.isdigit()
    and not re.search(r'\d', w)
]

counts = Counter(filtered_tokens)
final_tokens = [w for w in filtered_tokens if counts[w] >= 2]

vocab = sorted(list(set(final_tokens)))
vocab_size = len(vocab)
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for i, w in enumerate(vocab)}

print(f"حجم القاموس بعد حذف Stop Words: {vocab_size}")

حجم القاموس بعد حذف Stop Words: 3323


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [22]:
import tensorflow as tf


context_size = 2
X_new, y_new = [], []

for i in range(context_size, len(final_tokens) - context_size):
    ctx = [
        final_tokens[i - 2],
        final_tokens[i - 1],
        final_tokens[i + 1],
        final_tokens[i + 2]
    ]
    X_new.append([word_to_ix[w] for w in ctx])
    y_new.append(word_to_ix[final_tokens[i]])

X_tensor = tf.convert_to_tensor(X_new, dtype=tf.int32)
y_tensor = tf.convert_to_tensor(y_new, dtype=tf.int32)

Modeling  

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense

embed_dim = 16
input_length = 2 * context_size  # 2 * 2 = 4

model = Sequential([
    Input(shape=(input_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=embed_dim,
        name="cbow_embedding"
    ),

    GlobalAveragePooling1D(name="avg_pooling"),

    Dense(
        vocab_size,
        activation='softmax',
        name="output_dense"
    )
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ cbow_embedding (Embedding)      │ (None, 4, 16)          │        53,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ avg_pooling                     │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_dense (Dense)            │ (None, 3323)           │        56,491 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,659 (428.36 KB)

 Trainable params: 109,659 (428.36 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
pip install tensorflow

In [24]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [25]:
history = model.fit(
    X_tensor,
    y_tensor,
    epochs=75,
    batch_size=32,
    verbose=1
)

Epoch 1/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0109 - loss: 7.7398
Epoch 2/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.0195 - loss: 7.3390
Epoch 3/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.0311 - loss: 6.9820
Epoch 4/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.0474 - loss: 6.5468
Epoch 5/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0640 - loss: 6.1480
Epoch 6/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.0786 - loss: 5.8312
Epoch 7/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.0904 - loss: 5.5842
Epoch 8/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.0997 - loss: 5.3892
Epoch 9/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.1123 - loss: 5.2313
Epoch 10/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.1218 - loss: 5.0992
Epoch 11/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.1287 - loss: 4.9867
Epoch 12/75
864/864 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step

In [33]:

import numpy as np

embedding_matrix = model.get_layer("cbow_embedding").get_weights()[0]

print(f"شكل مصفوفة التضمين المستخرجة: {embedding_matrix.shape}")

def get_most_similar(target_word, top_n=5):
    target_word = target_word.lower()

    if target_word not in word_to_ix:
        return f"الكلمة '{target_word}' غير موجودة في القاموس!"

    target_idx = word_to_ix[target_word]
    target_vec = embedding_matrix[target_idx]

    dot_products = np.dot(embedding_matrix, target_vec)
    norms = np.linalg.norm(embedding_matrix, axis=1) * np.linalg.norm(target_vec)
    similarities = dot_products / (norms + 1e-9)

    best_indices = np.argsort(similarities)[::-1]

    results = []

    for idx in best_indices:
        if idx == target_idx:
            continue

        word = ix_to_word[idx]
        results.append((word, similarities[idx]))

        if len(results) == top_n:
            break

    return results


test_words = ["attack", "actor", "access", "malware", "security"]

print("\n--- نتائج فحص التقارب الدلالي (Semantic Similarity) ---")

for word in test_words:
    if word in word_to_ix:
        print(f"\nأقرب الكلمات إلى [{word}]:")

        for sim_word, score in get_most_similar(word, top_n=4):
            print(f"  - {sim_word:<15} (درجة التشابه: {score:.4f})")
    else:
        print(f"\nالكلمة [{word}] غير موجودة بالقاموس!")


شكل مصفوفة التضمين المستخرجة: (3323, 16)

--- نتائج فحص التقارب الدلالي (Semantic Similarity) ---

أقرب الكلمات إلى [attack]:
  - incidents       (درجة التشابه: 0.7488)
  - firm            (درجة التشابه: 0.7017)
  - test            (درجة التشابه: 0.6914)
  - prices          (درجة التشابه: 0.6729)

أقرب الكلمات إلى [actor]:
  - actors          (درجة التشابه: 0.7828)
  - expanded        (درجة التشابه: 0.7051)
  - asyncos         (درجة التشابه: 0.6816)
  - card            (درجة التشابه: 0.6757)

أقرب الكلمات إلى [access]:
  - identities      (درجة التشابه: 0.7402)
  - configuration   (درجة التشابه: 0.7284)
  - circular        (درجة التشابه: 0.7180)
  - layer           (درجة التشابه: 0.7056)

أقرب الكلمات إلى [malware]:
  - affiliated      (درجة التشابه: 0.7765)
  - sanctions       (درجة التشابه: 0.6796)
  - akamai          (درجة التشابه: 0.6714)
  - nexus           (درجة التشابه: 0.6585)

أقرب الكلمات إلى [security]:
  - usage           (درجة التشابه: 0.7681)
  - levels          (درجة الت

In [34]:
import numpy as np
import pickle

embedding_matrix = model.get_layer("cbow_embedding").get_weights()[0]
np.save("cbow_embedding_matrix.npy", embedding_matrix)

metadata = {
    "word_to_ix": word_to_ix,
    "ix_to_word": ix_to_word,
    "vocab": vocab
}

with open("cbow_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print(f"تم حفظ مصفوفة الأوزان بحجم: {embedding_matrix.shape}")
print(f"تم حفظ القاموس بعدد كلمات: {len(vocab):,}")

تم حفظ مصفوفة الأوزان بحجم: (3323, 16)
تم حفظ القاموس بعدد كلمات: 3,323


In [35]:
!pkill -f streamlit
!pkill -f cloudflared

!pip install -q streamlit plotly scikit-learn

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.2 MB/s eta 0:00:00


In [36]:
import numpy as np
import pickle

embedding_matrix = model.get_layer("cbow_embedding").get_weights()[0]
np.save("cbow_embedding_matrix.npy", embedding_matrix)

metadata = {
    "word_to_ix": word_to_ix,
    "ix_to_word": ix_to_word,
    "vocab": vocab
}

with open("cbow_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("تم حفظ الملفات بنجاح!")

تم حفظ الملفات بنجاح!


In [37]:
%%writefile app.py
import streamlit as st
import numpy as np
import pickle
import plotly.graph_objects as go
from sklearn.decomposition import PCA

st.set_page_config(
    page_title="CTI Embedding Explorer",
    page_icon="🛡️",
    layout="wide"
)

@st.cache_resource
def load_assets():
    matrix = np.load("cbow_embedding_matrix.npy")
    with open("cbow_metadata.pkl", "rb") as f:
        meta = pickle.load(f)
    return matrix, meta["word_to_ix"], meta["ix_to_word"], meta["vocab"]

try:
    embedding_matrix, word_to_ix, ix_to_word, vocab = load_assets()
except Exception as e:
    st.error(f"خطأ أثناء قراءة الملفات: {e}")
    st.stop()

def get_similar(query, top_n=6):
    query = query.lower().strip()
    if query not in word_to_ix:
        return None

    target_idx = word_to_ix[query]
    target_vec = embedding_matrix[target_idx]

    dot_products = np.dot(embedding_matrix, target_vec)
    norms = np.linalg.norm(embedding_matrix, axis=1) * np.linalg.norm(target_vec)
    similarities = dot_products / (norms + 1e-9)

    best_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in best_indices:
        token = ix_to_word[idx]
        if token != query:
            results.append((token, float(similarities[idx]), idx))
        if len(results) == top_n:
            break

    return results, target_idx

st.title("🛡️ Cyber Threat Intelligence (CTI) Embedding Explorer")
st.markdown("استكشاف الفضاء الدلالي لنموذج **CBOW** المدرّب على مقالات التهديدات السيبرانية.")

m1, m2, m3 = st.columns(3)
m1.metric("حجم القاموس", f"{len(vocab):,} كلمة")
m2.metric("أبعاد المتجه", f"{embedding_matrix.shape[1]}")
m3.metric("إجمالي العينات", f"{embedding_matrix.shape[0]:,}")

st.divider()

st.sidebar.header("⚙️ البحث والاستعلام")
search_word = st.sidebar.text_input("المصطلح المستهدف:", value="actor").strip().lower()
k_words = st.sidebar.slider("عدد الكلمات (Top-K):", min_value=3, max_value=20, value=6)

col_left, col_right = st.columns([1, 2])

query_output = get_similar(search_word, top_n=k_words)

with col_left:
    st.subheader("🔍 أقرب المصطلحات")
    if query_output is None:
        st.warning(f"الكلمة '{search_word}' غير موجودة بالقاموس.")
    else:
        neighbors, _ = query_output
        table_data = [{"المصطلح": w, "التشابه": f"{s:.4f}"} for w, s, _ in neighbors]
        st.table(table_data)

with col_right:
    st.subheader("📊 إسقاط الفضاء المتجهي (2D PCA)")
    if query_output is not None:
        neighbors, target_idx = query_output

        indices = [target_idx] + [idx for _, _, idx in neighbors]
        labels = [search_word] + [w for w, _, _ in neighbors]

        vectors = embedding_matrix[indices]
        pca = PCA(n_components=2, random_state=42)
        reduced = pca.fit_transform(vectors)

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=reduced[1:, 0],
            y=reduced[1:, 1],
            mode='markers+text',
            text=labels[1:],
            textposition="top center",
            marker=dict(size=14, color='#1f77b4'),
            name="المصطلحات المشابهة"
        ))

        fig.add_trace(go.Scatter(
            x=[reduced[0, 0]],
            y=[reduced[0, 1]],
            mode='markers+text',
            text=[f"★ {labels[0]}"],
            textposition="bottom center",
            marker=dict(size=20, color='#d62728', symbol='diamond'),
            name="الكلمة المستهدفة"
        ))

        fig.update_layout(
            height=460,
            margin=dict(l=20, r=20, t=30, b=20),
            xaxis=dict(title="PC 1"),
            yaxis=dict(title="PC 2"),
            showlegend=False
        )
        st.plotly_chart(fig, use_container_width=True)

Writing app.py


In [ ]:
!streamlit run app.py --server.enableCORS=false --server.enableXsrfProtection=false > /dev/null 2>&1 & cloudflared tunnel --url http://localhost:8501

In [ ]:
from google.colab import files
import zipfile

# 1. إنشاء ملف المتطلبات requirements.txt
requirements_txt = """streamlit>=1.30.0
numpy>=1.24.0
scikit-learn>=1.3.0
plotly>=5.18.0
tensorflow>=2.15.0
nltk>=3.8.1
beautifulsoup4>=4.12.0
requests>=2.31.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_txt)

# 2. تجميع الملفات في أرشيف ZIP
files_to_pack = [
    "app.py",
    "cbow_embedding_matrix.npy",
    "cbow_metadata.pkl",
    "requirements.txt"
]

zip_filename = "cti_cbow_project.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for f in files_to_pack:
        zipf.write(f)

print("جاري تحميل الملفات المضغوطة إلى جهازك...")
files.download(zip_filename)